# 🚀 TASK 6.6: FINAL LOCKED BENCHMARK & POST-HOC COMPARATIVE EVALUATION
### BÀI KIỂM CHUẨN KHÓA SỔ CHUNG CUỘC TRÊN 18 TẬP TEST & ĐỐI CHIẾU 3 BÊN MINH BẠCH

> **Mục tiêu**: Nạp checkpoint rút gọn `fatformer_crossdomain_weights_only.pth` (2.31 GB) cùng cặp tham số đóng băng $(T=1.0811, T^*=0.3700)$ từ Task 6.4 để đo đạc đối chiếu hậu nghiệm trên 18 tập test benchmark Task 5.1 và tính toán khoảng tin cậy 95% Bootstrap.
> **Môi trường**: Kaggle Server GPU / Colab (Internet: Off, 100% Offline).
> **Cơ chế tải kết quả**: Sau khi chạy xong (hoặc bấm **Save Version**), toàn bộ kết quả nằm trong tab **Output** của notebook trên Kaggle. Bạn có thể tải tệp `task_6.6_final_results.zip` hoặc xem các file CSV/PNG trực tiếp trên giao diện Kaggle.

## 🛠️ BƯỚC 1: THIẾT LẬP MÃ NGUỒN & TỰ ĐỘNG BUNG CÔNG CỤ ĐÁNH GIÁ (OFFLINE 100%)
> Giải nén mã nguồn nhẹ, tự động bung công cụ đánh giá `tools/eval_locked_benchmark.py` và `tools/plot_three_way_benchmark.py` mà không phụ thuộc vào gói code cũ.

In [ ]:
# ==============================================================================
# BƯỚC 1: THIẾT LẬP MÃ NGUỒN VÀ BUNG CÁC CÔNG CỤ ĐÁNH GIÁ (OFFLINE 100%)
# ==============================================================================
import os
import sys
import glob
import json
import shutil
import zipfile
import base64
import zlib

print("=" * 80)
print("BƯỚC 1: ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA TRÊN KAGGLE (OFFLINE 100%)")
print("=" * 80)

WORK_DIR = "/kaggle/working/fatformer-xla"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# Bộ lọc loại bỏ file nặng tránh tràn ổ đĩa
def ignore_heavy(dir_path, filenames):
    ignored = []
    for f in filenames:
        if f.endswith(('.pth', '.pt', '.ckpt', '.bin', '.tar', '.zip')):
            ignored.append(f)
        if f in ['checkpoints', 'pretrained', 'test_degraded']:
            ignored.append(f)
    return ignored

# 1. Tìm kiếm và sao chép mã nguồn nhẹ từ Kaggle Input
zip_cands = (
    glob.glob('/kaggle/input/**/fatformer_code.zip', recursive=True) +
    glob.glob('/kaggle/input/**/fatformer-code*.zip', recursive=True)
)
train_cands = glob.glob('/kaggle/input/**/train_crossdomain.py', recursive=True)

if zip_cands:
    print(f'[*] Giải nén mã nguồn từ gói zip: {zip_cands[0]}')
    with zipfile.ZipFile(zip_cands[0], 'r') as zf:
        zf.extractall(WORK_DIR)
elif train_cands:
    root_cand = os.path.abspath(os.path.join(os.path.dirname(train_cands[0]), '..', '..'))
    print(f'[*] Sao chép mã nguồn từ: {root_cand}')
    shutil.copytree(root_cand, WORK_DIR, dirs_exist_ok=True, ignore=ignore_heavy)
elif os.path.exists('/kaggle/input/fatformer-xla'):
    print('[*] Sao chép mã nguồn từ /kaggle/input/fatformer-xla...')
    shutil.copytree('/kaggle/input/fatformer-xla', WORK_DIR, dirs_exist_ok=True, ignore=ignore_heavy)
else:
    print('[*] Khởi tạo thư mục mã nguồn.')

# 2. Tự động bung công cụ eval_locked_benchmark.py mới nhất offline
os.makedirs(os.path.join(WORK_DIR, 'tools'), exist_ok=True)
eval_tool_path = os.path.join(WORK_DIR, 'tools', 'eval_locked_benchmark.py')
b64_eval_data = "eNrVPF1z20hy7/wVE2w5B3gpiJQlS8sKt0qWLNtnWXIkeutSDAsFAUMSSxLA4kMfp9NL/sLlIQ95uqf8htvH/BLnl6R7vgcEKW9yVam4yiQw093T09PT3dPT1Hdk5+UOibI4SWcDUlfTnSNs6TiO0zkLq7OsWNFi5w/nxwPyMaErcjKvw5R8nGchuc7wLZ2RkzqLyN+T0yyBhoTW5H1Yk4vZHOHdUVguyGv/tdfZIRdhTioK/4sM0MqMFHVFZlk6INOwmrKhgqjIyjLOVmGSBnc0mc2rMsjS5YOfV3MgcYqYNyF8REhsHq6QDhs1YqyJ8fYHZESGpO/3jvr9Lhm9hJee/+qw1wMioySdkyoD6AVMBGhVSQr0HsgPBy/ImyyryqoA6u6bIeD3eh6QzqA/gu8Eh4uyiiyzBCi9z3AyZGenomUVxPVq9QBAQGmBk6/mNcDNEkCsI3KTAcIf6rDi/K9CQEUWaEV+qUNycv0TuQ3J768vLwAAx8vYKnSSVZ4VFclK+VQ+qMcqWVH5/HOZpfJ5tsxu5HNYzPKwKGlnWmTA00NOSyK6ruF7SS/CFS3zMNIQoAwS5DSJqi45T0r4HNUA3SXH6UOXXOZVkqXhUvGX1qv8gYQlSXPFXVZEc+vFT1MGkjZb/WmdRpwiApwJVlhvXSXL0o/DKlRMwXNJgSN8OM/CmBYGwm1SAiEJi0IuUblKDvL5w7ns+rAKZ7TT+Xx1+fu3J6Pg6vISdSYr/Tys5n54U+K3K99/zpJUvcRJkYLY3CCYJksaBF6XOL7veF4nmRKLYArKAuoFi8YwBx0C/+Sbn6QlLSq317WQvA6SmYdlWFWFi8BlFWc1TNgpaJSl02RWF9TxNDHe7xu9Lk35th46bFs7XgtNWhRbaUL/MzQ7744vgusvb67fjq5BeGOG7eRFNgtTB2iX1cOSNp738OUmmbFmjhA9RAYYqCx/nIW1eIopzafhgjqdSadz+uHs7Mv1h8uWkWd1EtMYMZbxKtjr9YzHIJrO5GsfewTKElCCg16wd8jGZK/QD//td+znKHG4XFLszNN4hd+3v8TJdFqj4jEOj8/Pg/6RwZ4ppu/J2gQ6nU5Mp2SapDFoVAHGhN7DngNRu1GYxgloPy0HbB+O5dYbg5maTDyy8yOxmvgKgsqTiDDDpvBZB/4DTUC9jHQL/oN1hhFrqhqjADZBy5aIPJOS7GUcly5D8mzKBa3qIuX0VMcqrKI5Rfpornz8cKMuwEZ1USa3dDgqamqNJDBaaTd5FLDj3oSTEGAXWQpbvhMtw7Ikp2iw34TR4gZaXbBCn7K4XlLBPFhf2UdmSUiW4BPQEzRsO/gQ5o5S8u7zl12J4aPtRiq4rEGQpEkVBG5Jl9MumsqAMUDL4Z4hqbLOaeF6vgLXk0dEH9bnFqQFfJ7A017svuqS/usugS9Yd9DS4V6X5GHMdmi/gTyNOOp5ktKwcPuvyUvS7+/xT4sn2NSScVCiu7CIBd/3Bq9zoHYGtmFZu4o3997zLIC5f5vQO3ful8kfqdsDI7ljsCVWRHDnzj21Lswwo64LQ++Kb70wooHEEHrEIbjzMozmBB9gYSKIMgh49kgu27bFSCq6kjsLnR1uIebjJhPoLLlTH4D3zpY45XBZUnPJkHlGAjrZd6NLEMBe8WgDKPcEENpVwQKDiyqpO7ZU3QC4okym7t7efpfAh+d1N4GOshFNyww0ayPIBXyGSyS4omE6HPf8/aP+/sH+a9AueD44PNo7PGCPvaO9fu/w1cQm1foP/AdS2nt99Hrvh4N9RN973X/V2zs4Ys+HB4eHh/3+xGBrYuheECxpKpbJkLjQGuhztfAttBmtsFGvcGwqLnaBrDXuGPonqnsZ3tClWMuxw96ciWmDrGW1LdF3EFChBjJpS92zIO5xlVlQA7KPU9zAagEVIAUVGzRWqjES42U1A2psq/hZDuLgLLN4BC2gM/HYrsT4wrl698bx1kjcS0EoVXCBqg1H7yOaV+Qt+wInM2ilwif1RwrRe/ukxLLdd7mEhb+LQM3rigY3SRoWD8GKghmLyuAuqeZBBAp54zICaGgr2KR8FK7NXGkwVKAbuugKzGkIw1I4YCwzCLzVgYADzAtazrNlrLvFEYH5VG0MWO9E2R72/V4fOVI4fqAlghMHBPD5HJoAIQH6NCPprEb3MHrpW+glzI7GAZ8YDCwedk2uGSBEUswFn/llNq1W4b1roXZJnKyG0qpigBRIDPY9HoCLmPhRXoNfYSG6cCpCcEGa4+rxlwaYGJ/GSM01aP841LLz/LDEM4WbpJVAqfKgoCFuojT3y3rluoIIyNeDg6Jrjo1tnKO0gdWEEl48XAZhhL7MlePsEhQLR4fZeh76tF7P7yl2kPc2dvot7PQVOzZWE0oLXLPDMCQ7+NLCziqEI67A6fkH0OmqSX2vCHLyGWxeiDTlCIITxb5myROjolU0WluG/45cXZ7sHH85QV09/iymCr4GZ8p4Fk0pnbEmZK4jY0YOCIwTMHACBt60TQhriOBC1KkDGLLLPjvrRi0r4GjBZQvcwpatDP0yjEaYLjgU7IrqIVgmC+oyXBtmzNomkiAcrykThUFTI8AUAkTCGTPkxspqa1/r1VdIYBqEGGDh+MMO6tEu2RMzFWLABauh2ZXQTFzN5eBeq4yCFonstIkEe2D3C56JqQRjTUjPocpLTjWqVziRJr5SZabOzwL3DDlCoM5O2oyLHI2Xi1/fIxkNxpTBUF2JA+Jr48VT+wek1qK/wo88KvqO2lCOsOOuajEiC8fYSwrQaDNB5X5UcLLBBJI7VQGprWsAgR6ofhTrMsldtkXY5mDT8iz4fB08b4N+ajpQmbUKouT/xGd2hdlQjAzg7Mmwe7KzpDSWrft7TSfLkktjRlR63IbLZWm7ZzN2LQk72/WCYDH4ylY+MuTih7T4ZYj5MOaPtSH1tPWT/f9ADtYC0scnrqA3qAOgufBdi2+2q8ZwnOD/O/+/IgBMJASYSOB21VhiI6rGtgACaW4+hHxhFZIIMYTc4JgKR4yh8V7QfAmJx8ZJv6xv1Bz0hMZyjIkF2GoJNahprnggo6lvjmO09XwmlLE40LFMWzyzBqo1SDj4TQFNx+RneyzTGKRv8WOjroFqN6D5aY1oND+rRiAjoxhmB5XhVkrCtoYf5nBaid0Vj3OMTSN4Q9/HGzhLUQKqfoe6wAwj7xrDOrlAfO+AuVatkxOFNIfbg3asHw5bsJ5zL2BZgx8OwD67RVYD/5wtOOZAmKVacExs8iwrfVMnS9yv0QK27SqL6dKF3FS0yCGlXLHD2oAwI4gW33yP6S3sIGmp+dszWQkwc3A9A6cPuENY4dUIGM1bfJeXLnEo711I84JF5UnA2q0fcfMCpeeMX06IGsHKoHGzO6d4FpJJMh+z4k1b2ci7Wekwv8pcPlFpfzBrXxaRzwSn7i64TFkbj6vxKcDYCQTSuNhwtQqKQYfOyfmHz4OfktHO+W5/39E+WC3BUD3pTotV3VyXNCiLFTNjdussxBxuowNAA8jWB2i4+z04KrMer6OFPGVSPkVPt0BRx3jPBiJNYRkhtLRu5oh7ffWJvNr5SIsU5fM9eccG9bTomWxALIbMXC0vIWdY84ZSQj4tXsvu2iBeUz0MzlOWL5XgzHOD/g3IY4PGk6Ega5kOuAyoqEowwFaOmyx0wTrluLlCTE8MHXBhkJE3VXvI90enkdIYgbl/WxRZ8bcY0cpTQ4ibApkUNI9RQx8dVR6Tp8MFz9oddjOET+0ssO+xhWE4PwH06C7Gh4MJjrvw8e6kKjF/4iIaJLNBB9jhiyy8Abll3nzRhQc5Mk+Cud6TsUlxC7FB2fT5yAFOQU4H081R1ZTrKilh4rMgAmtYiSAK3Au7cCrQ3TsSYkEfSgdjIcM/1Sm9z2mEJ4LNBAygNhpKB//r3/884ffNc0wKQx4Mrrc/8dGHjxajT13yRVEdPjbZkNppH2DVQH83ISc4AlzZDtAsstgULV/4YCp/BfmgNb334b4cwCzrPGebPExw61p7l6+HYRxVp09vw6Vr3XJws8idD16lu2KfsotgPGbKS2H/uJjVK5pWn1mPi0fIIsm5Xttm5izBy9lz5sTIG7gJnK/CYmHc8AtB8TF8uIZA68KIu87Ojp48LBpGWkPh4aZhvayGzunx6BhuwnY1XLn7bWUBz4wrrXj7sMwD7PT3gdB2OkZYLimJw4qkJYsN5nSZD50LnRWMVdGCrE54ZigZlG4YSJy45EA8wwjGCMscfvNYWLhQQBzULh5HjjKC+64VuJ5ZFrG0J+kfgZpiPUf5jODA0+XM2RXbFx6OM+V2Sjd4pRfgGUJSAi0x5P96Kzb67rusWNCibEXvbcfWwWL74HjufFbQLKIC/DDiO6wEJ0MD8OlUCfpEWA5UHX51ZN83bhcRNw7tgo7qOHTQTXDPhq94kRHehgnk5JfUFY5C+DM2jIinxGjsi8UMOj3MYsKhA/H00ZEZwjhnx6Ozy6tPb694+dDZh4vjc3J+efLx7Sl58/bi5P2n46uPcGb5fHk92nl/eUJ+env14ezDyfEI7sXBrBxff7TMirC4o/WaH4hoMbiAkh+9R1n5zyOy6ptJDX9/+kT+REZyh/HKIAGnchsItTYsbuebZCCAuaCR1glfK3lD7LLg1huoweWy2xS1yMTNFZJTUQd/dY2RPJm+PWUBfWI6F8jkopUTDqDMlrfovBY5utC2igJ9qcgG0JSMTNT/whw/Q2W1v2ui8+DmBsW02m/i7y7C2WxJd5MUTMjuy5e/jQV18y9FAkL6RpGsh/6WpzCa4QAORgEu1ePdDRBSCFshJw1333JktJa2a09LHhX14XDYUD+lQH1fVPHdJMbVOWbLpD0XUosglVwGNw8BZArwrn2IOS4R4TaIs+uArDKa0aXI1sYRwgbyth8wYzB7uNNXUFNouBy4ZgNndFtr4wi7sMLwyz5wYrgL7GO0a5fkNGtIGpMdw9dEVRWZ/x6NG1ZIgjps/sEjwD8Fj8mTn6e8xojdHEO6k7wge09rVJCvRCfV9nv2peukJeT8jvxjTSstAVno4M55EWJMoY7wARQIBYdhKPfY3uYD2i+MHkSrpmBv2CGTxcuP9ko9/Y9Ei2km8P1GFRErpLNJdxHMFoEoUZJISQlEXEHMW7+E/g7SwyuuDUIQPKS2x1lD0zVI49aChpxNM8dJ6iKlrRNxXr50+HQ2lzI15mnPMV+Hm3RasFqLodalruugOk2V52npzlZuNkscBcO0PGBhxvJmiVIau06PJS5BCFjw4zp9ljd0MHGIr1af1dPTKKxt0j61qE2dBJtdzdE3SjvaMDtzlnxhA7iWSDD7n7Bis7twuXgWW1KYpojF0AfPFs4Aj9MUTt53rBgMUqX8PO860rb4P+fym/KHO3qTQ/Hp87TlysskrG3NLJHyWU9Tz7BmsMhP3rdZThzFSFxnFVyhTTFNinkNSD/j2f7WY9JhyYg1Qj6camsKkaZpcgBfdDPlXVj5jHUSIreBIhXj/Uh6kw3pgpE8t1fsPIUfjwbfT7y8rAD7+Mgul01mvKddw4iC6bR6n+z0kCUM6/rc9ILbMgvYnoaZsHHC0rOhmXN/AB5LViRvniOaLtM257/BpP9Gj/m385rf4Dm5uRShzp5PfkIBoQggvinBKyxrUfYPZwgzHP/nFN3iGzhExxBpKEgWIGFoz9bdWGItPIjD4Hhl2FK8UeZ3c1CwUFZ2h7yk0j2db5Q8mjhZabiu6UAUjSDLhzXLfBXmhnJfXYkgqiqH6xWYisa2CFMV0bGSfCCj6/PdWNbs61M8J6DfwW3O6+l0SXlikReliiM7hzUajFso5E3do443XQ+ODQdY9fCoBT+e8PHDqLNFSyt/lJAFswKSvw2riisFVWwPuFR8nu0VcffNXJ35zxgGrkL9sK4gmVxWAt7Hw3uX0BRP5fHQbEV7wQ/ym6w9pFtkthDqcTttgQGXlXQB+LsBfu3Lb8q8diR5QyywHgz5Uzj3ljS2RQplMZW++oPslF4ikX8IuVaZ19jGZkYM427XQhHtCkcf3uytJ5nVo3sWqLkZTVh1869T27wyETj5xpLF9Yl3zSk1KnF1hmLYTFk0IGWOYminLBSQnuAMLiXzAH+lAlw78PuDkiV/hJUxf4/Asz6n+gcMxk0ARLV49fpoceEw2mCn9Rg2mw63SgCBNsnuwj0sCgAwogBPaoq8AWwW9whRj43GRhmyWbmjoFVbE9io4FHAqq0JzCt5FBy+roHkFkS+BiD2SVBSpMXvjEUb3hkrYOsqBn2L1E2xHutHOoK1NONHvRyDH8qnCUH/OugflJim+oRCI8cnJxCcSCZ/pyT5u8nA35s+vQBAp7H9p84VSNDEkhI1kM5AbiaIlONWulCFaOKATA3w489WX6663EchM/Zeeo7KbLzyjZ/2Gb/XY/HcPIPc3CcaMtWHzcgelc6L90teEMYFDL88YuJmzqPgJwEeZnKHD5upGIutMGGGmW0y7n/w50C/DVnvv4muZgeLEgVYk+8ira7YcCxMYodM2GT8zVv7cQ8itP5Shv0Eplm9tG1/8+E2b3DFxNadjkE/TNlsnBhyAW63bX5VFMeEMS4sK2CTmTR/C7Fe0GfS0fbhOTJrJX8mGW05niPTKApUFJhNeRY5b8fNn0dtMz9iUcyutUWxTRPPUsK4Ae4PdIhKRdWG6YqtAN+f3oKjgZdrWYqAmLg5LEy1W7rmPpDo6tdyJg1Ru2mREZsL8MRGViQu4XIBAmttK/b9b6gnxHoWZkxuMkxAWjXZGyKaRgRihTUmalto0wxJLGRW0hYlZgSyVvPZzl+3bWCtG98WfGwKPIzaGM2QCNZ1g1HgAkVS7OSGOW4+JXZ2WauzgjwQK3tlRa8d+6jmQJFL653Tm69//cvFOzL6+uu/wtf7r7/+5TP5+OHrr//yiZy8//L1r/9xQT6+/88/H5NrgMAmgDr58vXXfzvZdOFkD4NpN6n67ZUvqG3Mw/Cugfa7GEUp5HbXi2X5NlTTJza7lVt0bAZxQ21hUO2xVgYR+XkGOdRGBkX3JgaFSm7hUWzZDUIU+G1suriJTz4MICCSygbZT97ZJbqtL9rgJ7QbopP10TZOV0OYMxbG5sDnP/zHHEjrD/3Zxi39FfgQSCiKKxJ9cQ5nQbw6CbKFkUjGCrKovG1NrJuokHmBwg0sp4f7kQhu2kTlRgCZ4SK594GGuu0FGwHVL1BeJ0MXNEPSrG7bAg1MVeMpIbdp5zZchHxGcbahZzKeY9Bzisd1FpGJCKerApmuHa50zeija0QQXSMM6HJf3mVeuWs7WB4FsnM++3GeWC2AunPwaN/4HT3+0QNDJFP/roBsi+t0Hb6qgnUPDR/YP/sWhvnqhhgaSYsmvRIrqsbzCU/AzpFA6xByV0J6jOVIT0Nyj5qs/noF5hL5LbxIqkFeDgJ3Md0nR9XVMs7wb1R8q8KKu0+tr5yGjzQEd/gIOcUHzMNYh1QHDg4hZpwg0GmEtpCFWkCrI8uWNtQ1OY3gCZMaUPy2wvCLZTlAglOW6XBe/NPOi9XOi5i8eD948WnwAkKdBrJR+6ST7DeQDmN/Q8K612W5avsSnx/PL3aP13jSjpqnHuAZ/vbBgGzPITjKebdhrTt2hqODDazZTyL8ewISx3D0myJ48Ojg1/G8jwZFmuCmnGRmkV1463nIBKOOQo3bdYhgYb0e1hda2R52KhfP3RYY/ZcbBto8tQEKayLBxOsGrljiX6aHMN7mRrSjD/jaOBi74xsNBIL6IJTcNfcAXNdAejaNoRgI/xQA/LAHVSMsoyQxazXtaw+5pYUv4o5JHZnZbjb421jFgn9dJGC5hyBg59kgYLUZgTMQv3vEMsTOfwNzMtLe"
with open(eval_tool_path, 'w', encoding='utf-8') as f:
    f.write(zlib.decompress(base64.b64decode(b64_eval_data.encode('ascii'))).decode('utf-8'))
print(f'[✓] Đã tạo script kiểm chuẩn: {eval_tool_path} ({os.path.getsize(eval_tool_path):,} bytes)')

# 3. Tự động bung công cụ plot_three_way_benchmark.py mới nhất offline
plot_tool_path = os.path.join(WORK_DIR, 'tools', 'plot_three_way_benchmark.py')
b64_plot_data = "eNqNWMtu48gV3esrLtgwQE5LHJJ6tCRAC1vuTidpP2JrJgEcgSiRJZExVUWQRbc0g1n1YpbJYJBFkCwyCbJIgCyCLALYiyw8P6J8SW4VH6JkudG0JJJV91W3bp1T5RfQ+qwFHvdDthhCJuatvmxpaJrWeEPEG54sadL61bvjIXxJ4SSkGZxy/IQwDuRLG04oA30SJJS2fknWMObLmCREhHdUdnnBkiS3BkqQ9BZ6Zs9oXHNICQtAJJkHfkjZsGGbcEJSGoWMwk+4B+MvL6/AsZwO6Ge20XBMOOMQhKj0BkVEhmKXEWFwXNjtmjZ8DmeO0WhvRU8JnKF1mHCUPJVP+lkY0VRwVO9J+Q68IVE0I96toQbcCJcxTwTwtHxK19Ujy5bxGkgKLC6blkTEERdRODPjtXyS3XEkGo3Lq4ufvR5P3KuLiwmM0KAZExGYZJbKu16+/4aHrHrxw4SRJdVdd45huq7RBM00NcNoXHwxccdvj68m7uXx5G3NntKv+0KV0+PJ8fXriYaPEV+k8i4wRy7m3hVymtz3ZO16aprClDMzZgvNwLHPISApESLRcdRmKnyeCVROqMfZPFxkCdWMYQPw2vabtV4dZ1vV0UhTdSSNNnw6xzRhlIUqZgdV1xE1s5TqWkrJjCesdWe5/db7IBR0kYS+BhjM832sZoXckTAis4gCjVIKGvojWSTQt/SGgTVBJyu7CWTlGJg5pZnN5GSlOjY7TSmUhl/RkW73mlihXcy7H4ejtmXhAKSVF4D1eZ0rgT3EUk2w+oksj6KUu+AFoXzzUCLioVLzCEbMk5Cm6PdGNclLG0cUC/KMeAmH4/H410yfbB5+zxYggs3DB0Nr7ou+Ibe0kDwPNvf/ZLhqNg/fyqKez7M05OyA0hUlUaF0srn/KwfBH//MYLl5+C0wyg9pXIxbx19IhSs5pgX6+GMIcfD4Fwbi8Qcv2NH58Xebhz/ABCdjjCGt4fX4NWqO0YcayP3fYvDyl83Dd6JUneYZXdpu6vEkz0zXMi2rCfmvbeX3rm32cTo6fdN+NVVzgA4g9UhEfVihFPgUEGK8IFMOCf76HI5y807NfH9gOmip75gOTvagZ7bR/KBr9nH2u2a3N81VOjWVQS6Lmm0UGnTMQU9p9rtYMOZgUIxihbIsNhHt2ILqEWX6dsqNvADfh74IUMwynW6uhAtGpDY2YVWaM5LoK2jlYs1tWpplC5Y2jUbaFg8rmJTAiGvb4xFPRtqLvif/sIH6C1o2dtVVrAXl2Kk7bm4Tte+wQNcKbfdgtua4bfedmb/n2Op37YFXd9zeGfHLasSdZwLY4R6J1PrL0zAVYRTByzFnaT0E2uvZlr0XwgzXr2Vp1RL+KcOKJkg6IX6pXEJY47hcc5zgSR6mBJebfIqaRcaKe3s6rIq/FJfSqnPbJS854bLZXFDhBjRcBEI3diRkJghjXGC96HPt62Bo2vNvjmrr68m1Wo/0yuhKNzCH1atKHTZ9LuEsMD5qRdCVGOm4BNofk5NSHueJn440Pp+nVECMXCPSj4UYkJHmUSZognNxhy8zLgRf4suco6rE2FdmN397r9IiRSK/nCSZFfTkilBEyA0SYb5T+4zN/X8zmCSbh395iHkK+nDf8fgPNtyuh7u03BLgE9bLz4PHfxO4RmSt+7edA+6bEBMfu4ydKFYi9G5TfXWgVdVoWlvtNQ+DZ0ZYN7JWBnQNcf8/8A7HA/qRsROm/Qk2wqWcRtvubjsiuqDM1yPuIQfHMU0gonNRtzzA5wR3GSSKcbosc1AtEGfLcc4QrohfsdxMbvTkRg1mGW7OWIDbPCYBN8q8wrWDG4Elv6N6GYsEGiRWk/i+W/BtQbf4iXlEktEkyWjhPZHeXMznx6myzj8V91X0+CyjPe04vqy3Kaq8RBCiyR1uXDnbUZDbGx+3NgiCdgtJaEtl8vccA5bIvx1BkQG2iHIyYbgs5xEnQj834DNw8IukEYcKRZiCEMUf58a0rvlyVDzdDO1pOUfnCIkkwnmEJUUk81JkdbBaSId5Gt2l/YRRCybtOorxSn7VbaQdy+z07VcyKmyeliacA6yZ0yVSoNOvOLQwYXW7vX0TnQMsmtOnvDkVp5YmnMGgMrEdCaZA3lUCqtjyRqfe2CkaO9tUyYpUNZfnsKkMIbkgUiiwHNlymcpXtY0cacNDXPqEfRXpGs96cHY99Hc8tFqHWHOXb3XJrc/b79TtO/v2DzFiYX4HD0GvTj01X3jmiPZ8PbFWgoaEykpRIXZABXFzotDz6kYuMg7I4CEHWQvXmN6yd/sLvC2LvmVPD/QXyLtdbZ+MvLmRde7kxsH67eC3h99+vu2cHpAs3N1ojnUkD1Kd/NbLb/38hrpH2nSbrp66sEdF1d8zW8K21d1LT056J0hvHzKQ3Pd9AcInhMMvsscfBJz/+AFR953iwWuSwVvJhRmidLa5/zv7VJ6rRfSULxIpj5KzGV+5grsET/A4Kt022xi0aRufwCXyjCWkHTciazwkFqSAh9YlQjVWQPrkzLt7wkUndIW7PZffFixRHR3JHUVa2ZOvDmxF3CEGTXHnooLQttpexNOSouIEtzO497r535++n8r/FawyUlIdx08oj3TqHxwzKiF6CF/vOv1GnW/xoOq6cgiuC6MRaK4rT7uuq+Vbwvzo2/g/FeciVw=="
with open(plot_tool_path, 'w', encoding='utf-8') as f:
    f.write(zlib.decompress(base64.b64decode(b64_plot_data.encode('ascii'))).decode('utf-8'))
print(f'[✓] Đã tạo script vẽ biểu đồ: {plot_tool_path} ({os.path.getsize(plot_tool_path):,} bytes)')

# 4. Xử lý vocab BPE tokenizer offline nếu có
clip_dir = os.path.join(WORK_DIR, 'src', 'models', 'clip')
os.makedirs(clip_dir, exist_ok=True)
target_bpe = os.path.join(clip_dir, 'bpe_simple_vocab_16e6.txt.gz')
bpe_cands = glob.glob('/kaggle/**/bpe_simple_vocab_16e6.txt.gz', recursive=True)
if not os.path.exists(target_bpe) and bpe_cands:
    shutil.copy2(bpe_cands[0], target_bpe)
    print(f'[✓] Đã đồng bộ BPE vocab offline: {target_bpe}')

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('[✓] Hoàn tất Bước 1: Môi trường sẵn sàng 100% Offline!')


## 🔍 BƯỚC 2: ĐỊNH VỊ CHECKPOINT KHÓA SỔ, CLIP VÀ 18 TẬP TEST BENCHMARK
> Tự động dò tìm các tệp checkpoint .pth, CLIP backbone và thư mục dữ liệu kiểm thử.

In [ ]:
# ==============================================================================
# BƯỚC 2: ĐỊNH VỊ CHECKPOINT, CLIP VÀ TẬP DỮ LIỆU TEST
# ==============================================================================
def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

def find_dir(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        for m in matches:
            if os.path.isdir(m):
                return os.path.abspath(m)
    return None

print("=" * 80)
print("BƯỚC 2: ĐỊNH VỊ CHECKPOINT VÀ DỮ LIỆU BENCHMARK")
print("=" * 80)

# Dò tìm checkpoint
ckpt_path = find_file([
    '/kaggle/input/**/fatformer_crossdomain_weights_only.pth',
    '/kaggle/input/**/crossdomain_model_best_m4.pth',
    '/kaggle/input/**/fatformer_plan_a_best_weights.pth',
    f'{WORK_DIR}/DATASET/checkpoints/fatformer_crossdomain_weights_only.pth',
    f'{WORK_DIR}/DATASET/checkpoints/m4/crossdomain_model_best_m4.pth'
])
print(f'[1] Checkpoint Khóa Sổ: {ckpt_path}')

# Dò tìm CLIP
clip_path = find_file([
    '/kaggle/input/**/ViT-L-14.pt',
    f'{WORK_DIR}/DATASET/pretrained/ViT-L-14.pt',
    f'{WORK_DIR}/pretrained/ViT-L-14.pt',
    '/kaggle/working/**/ViT-L-14.pt'
])
print(f'[2] CLIP ViT-L/14 Backbone: {clip_path}')

# Dò tìm thư mục test benchmark hoặc giải nén nếu có file tar
test_root = find_dir([
    '/kaggle/input/**/test_benchmark_gans',
    '/kaggle/input/**/test_benchmark',
    '/kaggle/input/**/CNN_synth_testset',
    '/kaggle/input/**/test_clean',
    f'{WORK_DIR}/DATASET/test_benchmark'
])

# Nếu chưa có thư mục giải nén nhưng có tệp .tar thì tự giải nén
if not test_root:
    tar_cands = glob.glob('/kaggle/input/**/test_benchmark*.tar', recursive=True) + \
                glob.glob('/kaggle/input/**/test_clean*.tar', recursive=True)
    if tar_cands:
        import tarfile
        ext_target = '/kaggle/working/test_benchmark'
        os.makedirs(ext_target, exist_ok=True)
        print(f'[*] Tự động giải nén dữ liệu kiểm thử từ: {tar_cands[0]}...')
        with tarfile.open(tar_cands[0], 'r') as tf:
            tf.extractall(ext_target)
        test_root = ext_target

print(f'[3] Thư mục Dữ Liệu Kiểm Thử: {test_root or "[Không tìm thấy tập test -> Chạy chế độ Dummy]"}')


## ⚡ BƯỚC 3: THỰC THI KIỂM CHUẨN KHÓA SỔ & TÍNH KHOẢNG TIN CẬY 95% BOOTSTRAP
> Áp dụng cố định $T=1.0811, T^*=0.3700$ và tính toán thống kê $B=1.000$ lần lấy mẫu lại (Bootstrap Resampling).

In [ ]:
# ==============================================================================
# BƯỚC 3: CHẠY KIỂM CHUẨN KHÓA SỔ (LOCKED BENCHMARK)
# ==============================================================================
import subprocess
import torch

OUTPUT_LOGS = '/kaggle/working/output_logs_task_6.6'
os.makedirs(OUTPUT_LOGS, exist_ok=True)

device_mode = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[*] Thiết bị tính toán: {device_mode}')

cmd = [
    sys.executable, '-u', f'{WORK_DIR}/tools/eval_locked_benchmark.py',
    '--temperature', '1.0811',
    '--threshold', '0.3700',
    '--output_dir', OUTPUT_LOGS,
    '--batch_size', '16',
    '--num_workers', '0',
    '--n_bootstrap', '1000',
    '--device', device_mode
]

if ckpt_path:
    cmd.extend(['--checkpoint', ckpt_path])
if clip_path:
    cmd.extend(['--clip_path', clip_path])
if test_root and os.path.exists(test_root):
    cmd.extend(['--test_root', test_root])
else:
    cmd.append('--test_dummy')

print('=' * 80)
print(f'[*] Lệnh thực thi: {" ".join(cmd)}')
print('=' * 80)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Tiến trình kết thúc với mã lỗi: {process.returncode}')
else:
    print('\n[✓] HOÀN TẤT VÒNG ĐO ĐẠC KIỂM CHUẨN CHUNG CUỘC!')


## 📦 BƯỚC 4: TỔNG HỢP ĐỐI CHIẾU 3 BÊN & XUẤT OUTPUT KAGGLE (CHUẨN DOWNLOAD)
> Tự động vẽ biểu đồ 300 DPI, hiển thị ma trận, đóng gói file `.zip` và copy các file chính ra thư mục `/kaggle/working/` để tải và xem trực tiếp trên tab **Output** của Kaggle.

In [ ]:
# ==============================================================================
# BƯỚC 4: BẢNG TỔNG HỢP ĐỐI CHIẾU 3 BÊN VÀ ĐÓNG GÓI OUTPUT KAGGLE
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

res_csv = os.path.join(OUTPUT_LOGS, 'final_posthoc_benchmark_matrix.csv')
res_json = os.path.join(OUTPUT_LOGS, 'final_locked_benchmark_report.json')

if os.path.exists(res_csv):
    df = pd.read_csv(res_csv)
    display(Markdown('### 📊 KẾT QUẢ KIỂM CHUẨN KHÓA SỔ TRÊN 18 TẬP TEST (TASK 6.6):'))
    display(df)

# Vẽ biểu đồ 3 bên nếu có script
plot_script = f'{WORK_DIR}/tools/plot_three_way_benchmark.py'
chart_generated = False
if os.path.exists(plot_script):
    try:
        subprocess.run([sys.executable, plot_script], check=True)
        chart_p = f'{WORK_DIR}/DATASET/logs/task_6.6_three_way_comparison.png'
        if os.path.exists(chart_p):
            shutil.copy2(chart_p, os.path.join(OUTPUT_LOGS, 'task_6.6_three_way_comparison.png'))
            chart_generated = True
            print('[✓] Đã tạo biểu đồ đối chiếu 3 bên: task_6.6_three_way_comparison.png')
    except Exception as e:
        print(f'[!] Cảnh báo vẽ biểu đồ: {e}')

# 1. Đóng gói toàn bộ kết quả vào tệp zip tại thư mục gốc /kaggle/working/
export_zip = '/kaggle/working/task_6.6_final_results.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_LOGS):
        for fn in files:
            fp = os.path.join(root, fn)
            arcname = os.path.relpath(fp, OUTPUT_LOGS)
            zf.write(fp, arcname)

# 2. Copy các file chính ra thẳng /kaggle/working để Kaggle Web Preview được trực tiếp
direct_exports = [
    ('final_posthoc_benchmark_matrix.csv', os.path.join(OUTPUT_LOGS, 'final_posthoc_benchmark_matrix.csv')),
    ('final_locked_benchmark_report.json', os.path.join(OUTPUT_LOGS, 'final_locked_benchmark_report.json')),
    ('task_6.6_three_way_comparison.png', os.path.join(OUTPUT_LOGS, 'task_6.6_three_way_comparison.png'))
]
for fname, fpath in direct_exports:
    if os.path.exists(fpath):
        target_dest = os.path.join('/kaggle/working', fname)
        shutil.copy2(fpath, target_dest)

# 3. Dọn dẹp mã nguồn tạm để tab Output chỉ hiển thị các tệp kết quả sạch sẽ
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
if os.path.exists(OUTPUT_LOGS):
    shutil.rmtree(OUTPUT_LOGS)

z_mb = os.path.getsize(export_zip) / (1024 * 1024) if os.path.exists(export_zip) else 0

print('=' * 80)
print('HOAN TAT DONG GOI OUTPUT KAGGLE (SAN SANG TAI VE)!')
print('=' * 80)
print(f'1. Tep nen tong hop: /kaggle/working/task_6.6_final_results.zip ({z_mb:.2f} MB)')
print('2. Cac tep xem truc tiep tren tab Output:')
for fname, _ in direct_exports:
    f_on_root = os.path.join('/kaggle/working', fname)
    if os.path.exists(f_on_root):
        print(f'   - {fname} ({os.path.getsize(f_on_root):,} bytes)')
print('\n-> BAN CHI CAN VAO MUC OUTPUT CUA NOTEBOOK TREN KAGGLE DE TAI TEP ZIP VE MAY!')
